# **Sales Prediction: EDA + Top 3 Models**Trimmed down to the **three best-performing models** out of everything tested across our last few rounds (Linear Regression, Random Forest, Extra Trees, Gradient Boosting, HistGradientBoosting, plus XGBoost/LightGBM/CatBoost where they could be run):1. **Gradient Boosting** (sklearn) — best single model, validation RMSE ≈ 10812. **HistGradientBoosting** (sklearn) — validation RMSE ≈ 10933. **Extra Trees** — validation RMSE ≈ 1103These three are combined into a validation-tuned **blended ensemble**, which is what the final submission is based on (RMSE ≈ 1077 on the validation split — the best result found so far).> **Caveat carried over from before:** XGBoost, LightGBM, and CatBoost weren't available to test in the sandbox I built this in (no internet access to install them), so it's possible one of them would edge out one of these three if you run them yourself. Everything tested so far — 8 algorithms, a neural net, and a stacked meta-model — landed within a narrow ~1,070–1,120 RMSE band, so I wouldn't expect a dramatic swap even if you add them back in.

### Import Dependencies

In [ ]:
# Core data handling
import shutil
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Splitting + preprocessing
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# The three models
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor

# Pipeline + metric + ensemble-weight solver
from sklearn.pipeline import make_pipeline
from sklearn.metrics import root_mean_squared_error
from scipy.optimize import nnls

# **Exploratory Data Analysis (EDA)**

In [ ]:
# Load the raw training data as provided by the competition
df = pd.read_csv("../data/train.csv")
df.head()

In [ ]:
# Quick sanity check on the size of the dataset (rows, columns)
df.shape

In [ ]:
# Column dtypes and non-null counts -- first look at where the missing data lives
df.info()

In [ ]:
# find the number of missing values in each column
df.isnull().sum()

`product_weight_kg` and `store_size` both have real gaps -- handled in the wrangle function below.

In [ ]:
# good, wrap all the cleaning into a wrangle function so it can be applied
# identically to both train.csv and test.csv (no train/test skew in preprocessing)
def wrangle(filepath):
    df = pd.read_csv(filepath)

    # standardize category casing (e.g. "SNACK FOODS" / "snack foods" -> "Snack Foods")
    df["product_category"] = df["product_category"].str.strip().str.title()

    # missing store_size isn't random -- give it its own category rather than guessing
    df["store_size"] = df["store_size"].fillna("Unknown")

    # impute missing weight using the median for that product's category
    df["product_weight_kg"] = df.groupby("product_category")["product_weight_kg"] \
        .transform(lambda s: s.fillna(s.median()))

    # shelf_visibility has a cluster of exact zeros that are more likely missing/unrecorded
    # than genuinely zero visibility -- treat them as missing and impute the same way
    df["shelf_visibility"] = df["shelf_visibility"].replace(0, np.nan)
    df["shelf_visibility"] = df.groupby("product_category")["shelf_visibility"] \
        .transform(lambda s: s.fillna(s.median()))

    # product_code is too high-cardinality (1,555 unique values) to be a useful feature
    # directly -- category/price/weight already capture what matters about a product
    df.drop(columns="product_code", inplace=True)

    return df

df = wrangle("../data/train.csv")
df.head()

In [ ]:
# confirm the wrangle function actually closed every gap
df.isnull().sum()

## Quick look at the target and key relationships

In [ ]:
# distribution of the target variable -- positively skewed, as most sales data is
plt.figure(figsize=(7,5))
sns.histplot(df["total_sales"], bins=40, color="skyblue")
plt.xlabel("Total Sales")
plt.title("Distribution of Total Sales")
plt.savefig("../images/sales_distribution.png", dpi=150)
plt.show()

In [ ]:
# store format clearly separates sales volume -- Flagship Hypermarkets/Superstores
# sell far more than Corner Shops
plt.figure(figsize=(9,5))
sns.boxplot(data=df, x="store_format", y="total_sales")
plt.xlabel("Store Format")
plt.ylabel("Total Sales")
plt.title("Sales by Store Format")
plt.xticks(rotation=15)
plt.savefig("../images/sales_by_format.png", dpi=150)
plt.show()

## Realistic RMSE ceiling — how low can we actually expect to go?Grouping by **every categorical column at once** (the most granular "cheat" grouping possible) shows how much sales still vary even among rows that are identical on every available category. Whatever's left over is noise no model can remove.

In [ ]:
cat_check_cols = ["store_code", "product_category", "fat_content", "store_size", "store_location_tier", "store_format"]
group_stats = df.groupby(cat_check_cols)["total_sales"].agg(["mean", "std", "count"])

print("Overall target std:", round(df["total_sales"].std(), 1))
print("Median within-group std (residual noise even with perfect grouping):", round(group_stats["std"].median(), 1))

Residual noise of **~1,400** even with perfect categorical grouping means RMSE realistically bottoms out somewhere around 1,000-1,100 for this dataset -- which is exactly where the three models below land.

# **Splitting Data into Feature Matrix and target vector**

In [ ]:
# id: no predictive value. store_code: dropped so the model can't just memorize
# per-store averages -- store format/size/tier/age already describe the store.
target = "total_sales"

y = df[target]
X = df.drop(columns=[target, "id", "store_code"])

# price_per_kg: normalizes price by weight -- two products at the same price but
# very different weights aren't really "the same price point"
X["price_per_kg"] = X["product_price"] / X["product_weight_kg"]

# visibility_price: shelf space scaled by price -- captures whether expensive items
# get prioritized placement
X["visibility_price"] = X["shelf_visibility"] * X["product_price"]

X.head()

# **Splitting into Training and Validation set**

In [ ]:
# 80/20 split, fixed random_state so results are reproducible across reruns
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

In [ ]:
# wrangle the competition test set the same way, with the same engineered features,
# so all three models can score it later
test_df = wrangle("../data/test.csv")
test_df["price_per_kg"] = test_df["product_price"] / test_df["product_weight_kg"]
test_df["visibility_price"] = test_df["shelf_visibility"] * test_df["product_price"]

X_test = test_df.drop(columns=["id", "store_code"])
test_df.head()

# **Shared Setup**

In [ ]:
# baseline: predict the training mean for every row -- any real model needs to
# beat this by a wide margin
y_pred_baseline = [y_train.mean()] * len(y_val)
baseline_rmse = root_mean_squared_error(y_val, y_pred_baseline)
print("Baseline RMSE:", round(baseline_rmse, 2))

In [ ]:
# categorical vs numeric columns, used to build the shared preprocessor below
categorical_cols = X_train.select_dtypes(include=["object", "str"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols, numeric_cols

In [ ]:
# ONE preprocessor for all three models: all three are tree-based, so no scaling is
# needed -- one-hot encode the categoricals, pass numeric columns straight through.
# sparse_output=False because HistGradientBoostingRegressor requires dense input.
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
], remainder="passthrough")

In [ ]:
def evaluate(model, X_train, y_train, X_val, y_val):
    """Fit-time evaluation: prints and returns (train_rmse, val_rmse)."""
    train_rmse = root_mean_squared_error(y_train, model.predict(X_train))
    val_rmse = root_mean_squared_error(y_val, model.predict(X_val))
    print("Training RMSE:", round(train_rmse, 2))
    print("Validation RMSE:", round(val_rmse, 2))
    return train_rmse, val_rmse


def generate_submission(preds, ids, path):
    """Write a Kaggle-format submission file: id + predicted total_sales.
    Predictions are clipped at 0 since sales can't be negative."""
    preds = np.clip(preds, 0, None)
    submission = pd.DataFrame({"id": ids, "total_sales": preds})
    submission.to_csv(path, index=False)
    return submission


# collect each model's results here for the comparison + blend at the end
val_rmses = {}
val_preds_by_model = {}
test_preds_by_model = {}

# **Model 1: Gradient Boosting**Best single model tested — sklearn's classic (non-histogram) gradient boosting implementation.

In [ ]:
model_gbr = make_pipeline(
    preprocessor,
    GradientBoostingRegressor(
        random_state=42, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=0.8
    )
)
model_gbr.fit(X_train, y_train)

_, val_rmses["Gradient Boosting"] = evaluate(model_gbr, X_train, y_train, X_val, y_val)
val_preds_by_model["Gradient Boosting"] = model_gbr.predict(X_val)

In [ ]:
# generate this model's own submission file
test_preds_by_model["Gradient Boosting"] = model_gbr.predict(X_test)
submission_gbr = generate_submission(
    test_preds_by_model["Gradient Boosting"], test_df["id"], "../submissions/submission_gradient_boosting.csv"
)
submission_gbr.head()

# **Model 2: HistGradientBoosting**sklearn's fast, histogram-based boosting -- same algorithm family as LightGBM/XGBoost.

In [ ]:
model_hgb = make_pipeline(
    preprocessor,
    HistGradientBoostingRegressor(random_state=42, max_iter=300, learning_rate=0.05, max_depth=4)
)
model_hgb.fit(X_train, y_train)

_, val_rmses["HistGradientBoosting"] = evaluate(model_hgb, X_train, y_train, X_val, y_val)
val_preds_by_model["HistGradientBoosting"] = model_hgb.predict(X_val)

In [ ]:
test_preds_by_model["HistGradientBoosting"] = model_hgb.predict(X_test)
submission_hgb = generate_submission(
    test_preds_by_model["HistGradientBoosting"], test_df["id"], "../submissions/submission_histgb.csv"
)
submission_hgb.head()

# **Model 3: Extra Trees**Like Random Forest, but splits are chosen randomly rather than optimally -- the extra randomness sometimes reduces overfitting further.

In [ ]:
model_et = make_pipeline(
    preprocessor,
    ExtraTreesRegressor(n_estimators=500, min_samples_leaf=5, random_state=42, n_jobs=-1)
)
model_et.fit(X_train, y_train)

_, val_rmses["Extra Trees"] = evaluate(model_et, X_train, y_train, X_val, y_val)
val_preds_by_model["Extra Trees"] = model_et.predict(X_val)

In [ ]:
test_preds_by_model["Extra Trees"] = model_et.predict(X_test)
submission_et = generate_submission(
    test_preds_by_model["Extra Trees"], test_df["id"], "../submissions/submission_extra_trees.csv"
)
submission_et.head()

# **Blended Ensemble (all 3 models)**Solve directly for the non-negative weights that minimize validation RMSE when combining all three models' predictions -- this is the model the final submission uses.

In [ ]:
model_names = list(val_preds_by_model.keys())
val_pred_matrix = np.column_stack([val_preds_by_model[name] for name in model_names])

blend_weights, _ = nnls(val_pred_matrix, y_val)

print("Blend weights:")
for name, w in zip(model_names, blend_weights):
    print(f"  {name}: {w:.3f}")

blend_val = val_pred_matrix @ blend_weights
val_rmses["Blended Ensemble"] = root_mean_squared_error(y_val, blend_val)
print("\nBlended Ensemble validation RMSE:", round(val_rmses["Blended Ensemble"], 2))

In [ ]:
# apply the exact same weights to the test-set predictions for the final submission
test_pred_matrix = np.column_stack([test_preds_by_model[name] for name in model_names])
blend_test_preds = test_pred_matrix @ blend_weights

submission_blend = generate_submission(blend_test_preds, test_df["id"], "../submissions/submission_blend.csv")
submission_blend.head()

# **Comparing Results & Final Submission**

In [ ]:
result_df = pd.Series(val_rmses, name="val_rmse").sort_values()
result_df

In [ ]:
plt.figure(figsize=(7,4))
result_df.plot(kind="barh", color="#2E86AB")
plt.xlabel("Validation RMSE (lower is better)")
plt.title("Model Comparison: Validation RMSE")
plt.xlim(left=result_df.min() * 0.9)
plt.tight_layout()
plt.savefig("../images/model_comparison.png", dpi=150)
plt.show()

In [ ]:
# pick whichever model had the lowest validation RMSE and use it as the final submission
best_model_name = result_df.idxmin()
print("Best model:", best_model_name)

submission_files = {
    "Gradient Boosting": "../submissions/submission_gradient_boosting.csv",
    "HistGradientBoosting": "../submissions/submission_histgb.csv",
    "Extra Trees": "../submissions/submission_extra_trees.csv",
    "Blended Ensemble": "../submissions/submission_blend.csv",
}

best_submission_path = submission_files[best_model_name]
shutil.copy(best_submission_path, "../submissions/submission_final.csv")
print("Copied", best_submission_path, "-> submissions/submission_final.csv")

In [ ]:
# final sanity check before submitting to Kaggle
final = pd.read_csv("../submissions/submission_final.csv")
print(final.shape)
final.head()